# Day 06: Analytical Window Functions Deep Dive
> **Part 6 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2006%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 🪟 Day 06: Analytical Window Functions Deep Dive
### 🎯 Learning Objectives:
1. Construct PySpark **Window Specifications** combining `partitionBy()` and `orderBy()`.
2. Master ranking functions:
   - `row_number()`: Strict sequential numbering (deduplication & first/last order identification).
   - `rank()` vs `dense_rank()`: Handling ties with vs without gaps.
3. Compute time-series inter-row metrics using `lag()` (previous order value) and `lead()` (subsequent order).
4. Calculate financial customer deltas (spend growth/churn per transaction).
5. Build cumulative financial metrics (running revenue total via `unboundedPreceding` to `currentRow`).
6. Calculate rolling moving averages using sliding frame windows (`rowsBetween(-2, 0)`).
7. Solve Top-N analytical problems (Top 2 orders per customer, Top revenue customers per city).
---


PySpark Window Functions
Today's goal:
Orders
   │
   ▼
Window Specification
   │
   ├── row_number()
   ├── rank()
   ├── dense_rank()
   ├── lag()
   ├── lead()
   ├── running total
   └── first/last value
   │
   ▼
Advanced Analytics

By the end, you'll be able to solve things like:
- Latest record per customer
- First order of each customer
- Previous order
- Next order
- Difference between current and previous order
- Customer running revenue
- Top 3 products per category
- Ranking customers within a city


### ⚙️ Initialize SparkSession for Day 06


In [194]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Day6_WindowFunctions")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [195]:
orders_data = [
    (1001, 101, "Rahul", "Kolkata", "Laptop", 1, 55000, "2026-09-01"),
    (1002, 102, "Amit", "Delhi", "Mouse", 2, 800, "2026-09-01"),
    (1003, 101, "Rahul", "Kolkata", "Keyboard", 1, 1500, "2026-09-03"),
    (1004, 103, "Sneha", "Mumbai", "Monitor", 2, 12000, "2026-09-03"),
    (1005, 101, "Rahul", "Kolkata", "Headphones", 2, 2500, "2026-09-05"),
    (1006, 102, "Amit", "Delhi", "Laptop", 1, 55000, "2026-09-06"),
    (1007, 104, "Priya", "Pune", "Laptop", 1, 55000, "2026-09-07"),
    (1008, 103, "Sneha", "Mumbai", "Mouse", 3, 800, "2026-09-08"),
    (1009, 101, "Rahul", "Kolkata", "Monitor", 1, 12000, "2026-09-09"),
    (1010, 102, "Amit", "Delhi", "Keyboard", 2, 1500, "2026-09-10"),
    (1011, 104, "Priya", "Pune", "Mouse", 2, 800, "2026-09-11"),
    (1012, 103, "Sneha", "Mumbai", "Laptop", 1, 55000, "2026-09-12"),
    (1013, 101, "Rahul", "Kolkata", "Laptop", 1, 55000, "2026-09-12"),
    (1014, 102, "Amit", "Delhi", "Monitor", 1, 12000, "2026-09-13"),
    (1015, 104, "Priya", "Pune", "Keyboard", 1, 1500, "2026-09-14")
]

columns = [
    "order_id",
    "customer_id",
    "customer_name",
    "city",
    "product",
    "quantity",
    "price",
    "order_date"
]

orders = spark.createDataFrame(
    orders_data,
    columns
)

In [196]:
orders = orders.withColumn(
    "order_date",
    to_date("order_date")
)

orders = orders.withColumn(
    "revenue",
    col("quantity") * col("price")
)

orders.show()

+--------+-----------+-------------+-------+----------+--------+-----+----------+-------+
|order_id|customer_id|customer_name|   city|   product|quantity|price|order_date|revenue|
+--------+-----------+-------------+-------+----------+--------+-----+----------+-------+
|    1001|        101|        Rahul|Kolkata|    Laptop|       1|55000|2026-09-01|  55000|
|    1002|        102|         Amit|  Delhi|     Mouse|       2|  800|2026-09-01|   1600|
|    1003|        101|        Rahul|Kolkata|  Keyboard|       1| 1500|2026-09-03|   1500|
|    1004|        103|        Sneha| Mumbai|   Monitor|       2|12000|2026-09-03|  24000|
|    1005|        101|        Rahul|Kolkata|Headphones|       2| 2500|2026-09-05|   5000|
|    1006|        102|         Amit|  Delhi|    Laptop|       1|55000|2026-09-06|  55000|
|    1007|        104|        Priya|   Pune|    Laptop|       1|55000|2026-09-07|  55000|
|    1008|        103|        Sneha| Mumbai|     Mouse|       3|  800|2026-09-08|   2400|
|    1009|

In [197]:
customer_window = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date")

Window
│
├── partitionBy()
│       ↓
│   Divide data into groups
│
└── orderBy()
        ↓
    Order rows inside each group


In [199]:
orders_with_row_number = orders.withColumn(
    "row_number",
    row_number().over(customer_window)
)

orders_with_row_number.select(
    "customer_id",
    "order_id",
    "order_date",
    "row_number"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+--------+----------+----------+
|customer_id|order_id|order_date|row_number|
+-----------+--------+----------+----------+
|        101|    1001|2026-09-01|         1|
|        101|    1003|2026-09-03|         2|
|        101|    1005|2026-09-05|         3|
|        101|    1009|2026-09-09|         4|
|        101|    1013|2026-09-12|         5|
|        102|    1002|2026-09-01|         1|
|        102|    1006|2026-09-06|         2|
|        102|    1010|2026-09-10|         3|
|        102|    1014|2026-09-13|         4|
|        103|    1004|2026-09-03|         1|
|        103|    1008|2026-09-08|         2|
|        103|    1012|2026-09-12|         3|
|        104|    1007|2026-09-07|         1|
|        104|    1011|2026-09-11|         2|
|        104|    1015|2026-09-14|         3|
+-----------+--------+----------+----------+



In [200]:
first_orders = (
    orders
    .withColumn(
        "rn",
        row_number().over(customer_window)
    )
    .filter(
        col("rn") == 1
    )
)

first_orders.select(
    "customer_id",
    "customer_name",
    "order_id",
    "order_date",
    "revenue"
).show()

+-----------+-------------+--------+----------+-------+
|customer_id|customer_name|order_id|order_date|revenue|
+-----------+-------------+--------+----------+-------+
|        101|        Rahul|    1001|2026-09-01|  55000|
|        102|         Amit|    1002|2026-09-01|   1600|
|        103|        Sneha|    1004|2026-09-03|  24000|
|        104|        Priya|    1007|2026-09-07|  55000|
+-----------+-------------+--------+----------+-------+



In [201]:
latest_window = Window \
    .partitionBy("customer_id") \
    .orderBy(col("order_date").desc())

In [202]:
latest_orders = (
    orders
    .withColumn(
        "rn",
        row_number().over(latest_window)
    )
    .filter(
        col("rn") == 1
    )
)

In [203]:
latest_orders.select(
    "customer_id",
    "customer_name",
    "order_id",
    "order_date",
    "revenue"
).show()

+-----------+-------------+--------+----------+-------+
|customer_id|customer_name|order_id|order_date|revenue|
+-----------+-------------+--------+----------+-------+
|        101|        Rahul|    1013|2026-09-12|  55000|
|        102|         Amit|    1014|2026-09-13|  12000|
|        103|        Sneha|    1012|2026-09-12|  55000|
|        104|        Priya|    1015|2026-09-14|   1500|
+-----------+-------------+--------+----------+-------+



In [204]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

In [205]:
revenue_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("revenue").desc())
)

ranked_orders = orders.withColumn(
    "revenue_rank",
    rank().over(revenue_window)
)

In [206]:
ranked_orders.select(
    "customer_id",
    "order_id",
    "revenue",
    "revenue_rank"
).orderBy(
    "customer_id",
    "revenue_rank"
).show()

+-----------+--------+-------+------------+
|customer_id|order_id|revenue|revenue_rank|
+-----------+--------+-------+------------+
|        101|    1001|  55000|           1|
|        101|    1013|  55000|           1|
|        101|    1009|  12000|           3|
|        101|    1005|   5000|           4|
|        101|    1003|   1500|           5|
|        102|    1006|  55000|           1|
|        102|    1014|  12000|           2|
|        102|    1010|   3000|           3|
|        102|    1002|   1600|           4|
|        103|    1012|  55000|           1|
|        103|    1004|  24000|           2|
|        103|    1008|   2400|           3|
|        104|    1007|  55000|           1|
|        104|    1011|   1600|           2|
|        104|    1015|   1500|           3|
+-----------+--------+-------+------------+



RANK vs DENSE_RANK
This is important.
Suppose revenues are:
55000
55000
12000
800

rank()
55000 → 1
55000 → 1
12000 → 3
800   → 4

dense_rank()
55000 → 1
55000 → 1
12000 → 2
800   → 3


In [208]:
rank_test = orders.withColumn(
    "rank",
    rank().over(revenue_window)
).withColumn(
    "dense_rank",
    dense_rank().over(revenue_window)
)

rank_test.select(
    "customer_id",
    "order_id",
    "revenue",
    "rank",
    "dense_rank"
).orderBy(
    "customer_id",
    "rank"
).show()

+-----------+--------+-------+----+----------+
|customer_id|order_id|revenue|rank|dense_rank|
+-----------+--------+-------+----+----------+
|        101|    1001|  55000|   1|         1|
|        101|    1013|  55000|   1|         1|
|        101|    1009|  12000|   3|         2|
|        101|    1005|   5000|   4|         3|
|        101|    1003|   1500|   5|         4|
|        102|    1006|  55000|   1|         1|
|        102|    1014|  12000|   2|         2|
|        102|    1010|   3000|   3|         3|
|        102|    1002|   1600|   4|         4|
|        103|    1012|  55000|   1|         1|
|        103|    1004|  24000|   2|         2|
|        103|    1008|   2400|   3|         3|
|        104|    1007|  55000|   1|         1|
|        104|    1011|   1600|   2|         2|
|        104|    1015|   1500|   3|         3|
+-----------+--------+-------+----+----------+



In [209]:
rank_test.select(
    "customer_id",
    "order_id",
    "revenue",
    "rank",
    "dense_rank"
).orderBy(
    "customer_id",
    col("revenue").desc()
).show()

+-----------+--------+-------+----+----------+
|customer_id|order_id|revenue|rank|dense_rank|
+-----------+--------+-------+----+----------+
|        101|    1001|  55000|   1|         1|
|        101|    1013|  55000|   1|         1|
|        101|    1009|  12000|   3|         2|
|        101|    1005|   5000|   4|         3|
|        101|    1003|   1500|   5|         4|
|        102|    1006|  55000|   1|         1|
|        102|    1014|  12000|   2|         2|
|        102|    1010|   3000|   3|         3|
|        102|    1002|   1600|   4|         4|
|        103|    1012|  55000|   1|         1|
|        103|    1004|  24000|   2|         2|
|        103|    1008|   2400|   3|         3|
|        104|    1007|  55000|   1|         1|
|        104|    1011|   1600|   2|         2|
|        104|    1015|   1500|   3|         3|
+-----------+--------+-------+----+----------+



In [210]:
# Now we get to the part you already know from SQL.
# lag() gives the previous row.
customer_date_window = Window \
    .partitionBy("customer_id") \
    .orderBy("order_date")

In [211]:
orders_lag = orders.withColumn(
    "previous_revenue",
    lag("revenue", 1).over(customer_date_window)
)

In [212]:
orders_lag.select(
    "customer_id",
    "order_id",
    "order_date",
    "revenue",
    "previous_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+--------+----------+-------+----------------+
|customer_id|order_id|order_date|revenue|previous_revenue|
+-----------+--------+----------+-------+----------------+
|        101|    1001|2026-09-01|  55000|            NULL|
|        101|    1003|2026-09-03|   1500|           55000|
|        101|    1005|2026-09-05|   5000|            1500|
|        101|    1009|2026-09-09|  12000|            5000|
|        101|    1013|2026-09-12|  55000|           12000|
|        102|    1002|2026-09-01|   1600|            NULL|
|        102|    1006|2026-09-06|  55000|            1600|
|        102|    1010|2026-09-10|   3000|           55000|
|        102|    1014|2026-09-13|  12000|            3000|
|        103|    1004|2026-09-03|  24000|            NULL|
|        103|    1008|2026-09-08|   2400|           24000|
|        103|    1012|2026-09-12|  55000|            2400|
|        104|    1007|2026-09-07|  55000|            NULL|
|        104|    1011|2026-09-11|   1600|           5500

In [213]:
# Revenue difference
# Now calculate:
# current revenue - previous revenue
orders_lag = orders_lag.withColumn(
    "revenue_difference",
    col("revenue") - col("previous_revenue")
)

orders_lag.select(
    "customer_id",
    "order_date",
    "revenue",
    "previous_revenue",
    "revenue_difference"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+----------------+------------------+
|customer_id|order_date|revenue|previous_revenue|revenue_difference|
+-----------+----------+-------+----------------+------------------+
|        101|2026-09-01|  55000|            NULL|              NULL|
|        101|2026-09-03|   1500|           55000|            -53500|
|        101|2026-09-05|   5000|            1500|              3500|
|        101|2026-09-09|  12000|            5000|              7000|
|        101|2026-09-12|  55000|           12000|             43000|
|        102|2026-09-01|   1600|            NULL|              NULL|
|        102|2026-09-06|  55000|            1600|             53400|
|        102|2026-09-10|   3000|           55000|            -52000|
|        102|2026-09-13|  12000|            3000|              9000|
|        103|2026-09-03|  24000|            NULL|              NULL|
|        103|2026-09-08|   2400|           24000|            -21600|
|        103|2026-09-12|  55000|  

In [214]:
# lead() gives the next row.
orders_lead = orders.withColumn(
    "next_revenue",
    lead("revenue", 1).over(customer_date_window)
)

orders_lead.select(
    "customer_id",
    "order_date",
    "revenue",
    "next_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+------------+
|customer_id|order_date|revenue|next_revenue|
+-----------+----------+-------+------------+
|        101|2026-09-01|  55000|        1500|
|        101|2026-09-03|   1500|        5000|
|        101|2026-09-05|   5000|       12000|
|        101|2026-09-09|  12000|       55000|
|        101|2026-09-12|  55000|        NULL|
|        102|2026-09-01|   1600|       55000|
|        102|2026-09-06|  55000|        3000|
|        102|2026-09-10|   3000|       12000|
|        102|2026-09-13|  12000|        NULL|
|        103|2026-09-03|  24000|        2400|
|        103|2026-09-08|   2400|       55000|
|        103|2026-09-12|  55000|        NULL|
|        104|2026-09-07|  55000|        1600|
|        104|2026-09-11|   1600|        1500|
|        104|2026-09-14|   1500|        NULL|
+-----------+----------+-------+------------+



Conceptually:
LAG

current
   ↑
previous


LEAD

current
   ↓
next


Running total 🔥🔥
Now calculate each customer's cumulative revenue.
This is extremely useful.
We need a window with:
PARTITION BY customer
ORDER BY date
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW


In [217]:
running_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

In [218]:
orders_running = orders.withColumn(
    "running_revenue",
    sum("revenue").over(running_window)
)

In [219]:
orders_running.select(
    "customer_id",
    "order_date",
    "revenue",
    "running_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+---------------+
|customer_id|order_date|revenue|running_revenue|
+-----------+----------+-------+---------------+
|        101|2026-09-01|  55000|          55000|
|        101|2026-09-03|   1500|          56500|
|        101|2026-09-05|   5000|          61500|
|        101|2026-09-09|  12000|          73500|
|        101|2026-09-12|  55000|         128500|
|        102|2026-09-01|   1600|           1600|
|        102|2026-09-06|  55000|          56600|
|        102|2026-09-10|   3000|          59600|
|        102|2026-09-13|  12000|          71600|
|        103|2026-09-03|  24000|          24000|
|        103|2026-09-08|   2400|          26400|
|        103|2026-09-12|  55000|          81400|
|        104|2026-09-07|  55000|          55000|
|        104|2026-09-11|   1600|          56600|
|        104|2026-09-14|   1500|          58100|
+-----------+----------+-------+---------------+



In [220]:
orders_running = orders_running.withColumn(
    "running_quantity",
    sum("quantity").over(running_window)
)

orders_running.select(
    "customer_id",
    "order_date",
    "quantity",
    "running_quantity"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+--------+----------------+
|customer_id|order_date|quantity|running_quantity|
+-----------+----------+--------+----------------+
|        101|2026-09-01|       1|               1|
|        101|2026-09-03|       1|               2|
|        101|2026-09-05|       2|               4|
|        101|2026-09-09|       1|               5|
|        101|2026-09-12|       1|               6|
|        102|2026-09-01|       2|               2|
|        102|2026-09-06|       1|               3|
|        102|2026-09-10|       2|               5|
|        102|2026-09-13|       1|               6|
|        103|2026-09-03|       2|               2|
|        103|2026-09-08|       3|               5|
|        103|2026-09-12|       1|               6|
|        104|2026-09-07|       1|               1|
|        104|2026-09-11|       2|               3|
|        104|2026-09-14|       1|               4|
+-----------+----------+--------+----------------+



In [221]:
moving_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(-2, 0)
)

In [222]:
orders_moving = orders.withColumn(
    "moving_avg_revenue",
    avg("revenue").over(moving_window)
)

In [223]:
orders_moving.select(
    "customer_id",
    "order_date",
    "revenue",
    "moving_avg_revenue"
).orderBy(
    "customer_id",
    "order_date"
).show()

+-----------+----------+-------+------------------+
|customer_id|order_date|revenue|moving_avg_revenue|
+-----------+----------+-------+------------------+
|        101|2026-09-01|  55000|           55000.0|
|        101|2026-09-03|   1500|           28250.0|
|        101|2026-09-05|   5000|           20500.0|
|        101|2026-09-09|  12000| 6166.666666666667|
|        101|2026-09-12|  55000|           24000.0|
|        102|2026-09-01|   1600|            1600.0|
|        102|2026-09-06|  55000|           28300.0|
|        102|2026-09-10|   3000|19866.666666666668|
|        102|2026-09-13|  12000|23333.333333333332|
|        103|2026-09-03|  24000|           24000.0|
|        103|2026-09-08|   2400|           13200.0|
|        103|2026-09-12|  55000|27133.333333333332|
|        104|2026-09-07|  55000|           55000.0|
|        104|2026-09-11|   1600|           28300.0|
|        104|2026-09-14|   1500|19366.666666666668|
+-----------+----------+-------+------------------+



Current row
+
previous 2 rows


In [225]:
top_orders_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(col("revenue").desc())
)

In [226]:
top_2_orders = (
    orders
    .withColumn(
        "rn",
        row_number().over(top_orders_window)
    )
    .filter(
        col("rn") <= 2
    )
)

In [227]:
top_2_orders.select(
    "customer_id",
    "order_id",
    "revenue",
    "rn"
).orderBy(
    "customer_id",
    "rn"
).show()

+-----------+--------+-------+---+
|customer_id|order_id|revenue| rn|
+-----------+--------+-------+---+
|        101|    1001|  55000|  1|
|        101|    1013|  55000|  2|
|        102|    1006|  55000|  1|
|        102|    1014|  12000|  2|
|        103|    1012|  55000|  1|
|        103|    1004|  24000|  2|
|        104|    1007|  55000|  1|
|        104|    1011|   1600|  2|
+-----------+--------+-------+---+



In [228]:
# Rank customers within their city
# Now we'll combine aggregation + window.
# First calculate customer revenue.
customer_summary = (
    orders
    .groupBy(
        "customer_id",
        "customer_name",
        "city"
    )
    .agg(
        sum("revenue").alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
)

customer_summary.show()

+-----------+-------------+-------+-------------+------------+
|customer_id|customer_name|   city|total_revenue|total_orders|
+-----------+-------------+-------+-------------+------------+
|        102|         Amit|  Delhi|        71600|           4|
|        104|        Priya|   Pune|        58100|           3|
|        103|        Sneha| Mumbai|        81400|           3|
|        101|        Rahul|Kolkata|       128500|           5|
+-----------+-------------+-------+-------------+------------+



In [229]:
city_window = (
    Window
    .partitionBy("city")
    .orderBy(col("total_revenue").desc())
)

In [230]:
customer_ranked = customer_summary.withColumn(
    "city_rank",
    dense_rank().over(city_window)
)

In [231]:
customer_ranked.orderBy(
    "city",
    "city_rank"
).show()

+-----------+-------------+-------+-------------+------------+---------+
|customer_id|customer_name|   city|total_revenue|total_orders|city_rank|
+-----------+-------------+-------+-------------+------------+---------+
|        102|         Amit|  Delhi|        71600|           4|        1|
|        101|        Rahul|Kolkata|       128500|           5|        1|
|        103|        Sneha| Mumbai|        81400|           3|        1|
|        104|        Priya|   Pune|        58100|           3|        1|
+-----------+-------------+-------+-------------+------------+---------+



In [232]:
top_city_customers = customer_ranked.filter(
    col("city_rank") == 1
)

top_city_customers.show()

+-----------+-------------+-------+-------------+------------+---------+
|customer_id|customer_name|   city|total_revenue|total_orders|city_rank|
+-----------+-------------+-------+-------------+------------+---------+
|        102|         Amit|  Delhi|        71600|           4|        1|
|        101|        Rahul|Kolkata|       128500|           5|        1|
|        103|        Sneha| Mumbai|        81400|           3|        1|
|        104|        Priya|   Pune|        58100|           3|        1|
+-----------+-------------+-------+-------------+------------+---------+



GROUP BY
    ↓
calculate metric
    ↓
WINDOW
    ↓
RANK
    ↓
filter rank = 1


---
### 🎓 Day 06 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 07: Modular Production Pipeline Architecture & Reusable ETL Functions**](./Day_07_Modular_Production_ETL_Architecture.ipynb)
